##1. csv 파일 전처리

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import os  # ✅ 추가

# CSV 파일 불러오기
df = pd.read_csv('/content/drive/MyDrive/OnSafe/modeling.csv')
print(f"원본 행 수: {len(df)}")

# 1. center_acceleration 컬럼 삭제
df = df.drop(columns=['center_acceleration'])
print(f"\n[1단계] center_acceleration 삭제 후 열 수: {df.shape[1]}")

# 2. [video, file_id, frame, timestamp] 기준 중복 행 제거
before = len(df)
df = df.drop_duplicates(subset=['video', 'file_id', 'frame', 'timestamp'])
df = df.reset_index(drop=True)
print(f"\n[2단계] 중복 제거 전 행 수: {before}")
print(f"        중복 제거 후 행 수: {len(df)}")
print(f"        삭제된 행 수: {before - len(df)}")

# 3. 한 행에서 0이거나 null인 값이 1/2 이상이면 해당 행 삭제
before = len(df)
threshold = df.shape[1] / 3
mask = df.apply(lambda row: (row.isna() | (row == 0)).sum() < threshold, axis=1)
df = df[mask]
df = df.reset_index(drop=True)
print(f"\n[3단계] 0/null 필터 전 행 수: {before}")
print(f"        0/null 필터 후 행 수: {len(df)}")
print(f"        삭제된 행 수: {before - len(df)}")

print(f"\n최종 행 수: {len(df)}, 열 수: {df.shape[1]}")

# 결과 저장
# output_dir = '/content/drive/MyDrive/OnSafe'
# output_filename = 'modeling_clean.csv'
# full_path = os.path.join(output_dir, output_filename)
# df.to_csv(full_path, index=False)

원본 행 수: 524697

[1단계] center_acceleration 삭제 후 열 수: 52

[2단계] 중복 제거 전 행 수: 524697
        중복 제거 후 행 수: 334282
        삭제된 행 수: 190415

[3단계] 0/null 필터 전 행 수: 334282
        0/null 필터 후 행 수: 297758
        삭제된 행 수: 36524

최종 행 수: 297758, 열 수: 52


In [ ]:
video_counts = df['video'].value_counts()

print(f"ADL 개수: {video_counts.get('ADL', 0)}개")
print(f"FALL 개수: {video_counts.get('FALL', 0)}개")

ADL 개수: 101656개
FALL 개수: 196102개


In [ ]:
# 결과 저장
output_dir = '/content/drive/MyDrive/OnSafe'
output_filename = 'modeling_clean_1.csv'
full_path = os.path.join(output_dir, output_filename)
df.to_csv(full_path, index=False)

##2. 선형보간 기법 추가

In [2]:
import pandas as pd
import numpy as np

In [17]:
df = pd.read_csv('/content/drive/MyDrive/OnSafe/modeling_clean_1.csv')

In [18]:
df.shape

(297758, 52)

In [19]:
# 1. 0값을 NaN으로 치환 (숫자형 컬럼만)
numeric_cols = df.select_dtypes(include='number').columns
df[numeric_cols] = df[numeric_cols].replace(0, np.nan)

In [20]:
# 2. 보간 처리를 위한 함수 정의
def interpolate_group(group):
    # 숫자형 컬럼만 선택
    num_cols = group.select_dtypes(include='number').columns

    for col in num_cols:
      series = group[col]

      # 2-1. 비선형(cubic) 보간 시도
      try:
        interpolated = series.interpolate(method='cubic', limit_direction='both')
        if interpolated.isna().any():
          raise ValueError("cubic 보간 후 NaN 잔존")
        group[col] = interpolated

      # 2-2. cubic 실패한 컬럼만 선형(linear) 보간으로 재처리
      except ValueError:
        group[col] = series.interpolate(method='linear', limit_direction='both')

    # 양 끝단 등 남은 NaN 처리
    return group.ffill().bfill()

In [21]:
# 3. file_id[:-8] 기준으로 그룹화하여 보간 실행
df['_group_key'] = df['file_id'].str[:-8]

df = (
    df.groupby('_group_key', group_keys=False)
    .apply(interpolate_group)
)

# 임시 그룹 키 컬럼 제거
df = df.drop(columns=['_group_key'])
df = df.reset_index(drop=True)

/tmp/ipykernel_3111/2706812936.py:6: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(interpolate_group)


In [22]:
df['Label'] = df['Label'].fillna(0)

print(f"Label NaN 개수: {df['Label'].isna().sum()}개")
print(f"Label 값 분포:\n{df['Label'].value_counts(dropna=False)}")

Label NaN 개수: 0개
Label 값 분포:
Label
1.0    196102
0.0    101656
Name: count, dtype: int64


In [23]:
print(f"보간 완료 행 수 : {len(df)}")
print(f"잔존 NaN 수 : {df.isna().sum().sum()}")

보간 완료 행 수 : 297758
잔존 NaN 수 : 7008


In [24]:
for video_type in ['ADL', 'FALL']:
    null_count = df[df['video'] == video_type].isnull().sum().sum()
    print(f"{video_type} NaN 개수: {null_count}개")

ADL NaN 개수: 30개
FALL NaN 개수: 6978개


In [25]:
before = len(df)
df = df.dropna()
df = df.reset_index(drop=True)

print(f"삭제 전 행 수: {before}개")
print(f"삭제 후 행 수: {len(df)}개")
print(f"삭제된 행 수: {before - len(df)}개")
print(f"잔존 NaN 수: {df.isna().sum().sum()}개")

삭제 전 행 수: 297758개
삭제 후 행 수: 296567개
삭제된 행 수: 1191개
잔존 NaN 수: 0개


In [26]:
import os

output_dir = '/content/drive/MyDrive/OnSafe'
output_filename = 'modeling_final.csv'
full_path = os.path.join(output_dir, output_filename)
df.to_csv(full_path, index=False)